In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import sys
import time
import sqlite3
import pandas as pd

# === 初期設定 ===
start_time = time.time()

# jupyter/py 両対応（必ず使用）
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR  # GSHEET_NAME等は不要

# パス定義（必ず使用）
if os.name == 'nt':
    user_base = os.path.join(os.environ["USERPROFILE"], "myenv310", PROJECT_DIR)
else:
    user_base = os.path.join(os.path.expanduser("~"), "myenv310", PROJECT_DIR)

db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# === 必須基本列 ===
BASE_NEEDED = ["実行日", "台番号", "大当り回数"]

with sqlite3.connect(db_path) as conn:
    # テーブル列一覧
    pragma = pd.read_sql_query("PRAGMA table_info(result_table);", conn)
    table_cols = set(pragma["name"].tolist())

    # 基本列チェック
    missing = [c for c in BASE_NEEDED if c not in table_cols]
    if missing:
        raise RuntimeError(f"result_table に必要列がありません: {missing}")

    # ステータス/スタート列（存在するものだけ採用）
    status_cols = [f"ステータス{i}回前" for i in range(1, 101) if f"ステータス{i}回前" in table_cols]
    start_cols  = [f"スタート{i}回前"   for i in range(1, 101) if f"スタート{i}回前"   in table_cols]

    # ステータス/スタートは同じ番号だけで評価するので、共通に存在する番号を抽出
    common_nums = [
        i for i in range(1, 101)
        if (f"ステータス{i}回前" in table_cols) and (f"スタート{i}回前" in table_cols)
    ]
    if not common_nums:
        raise RuntimeError("評価対象の『ステータスX回前』『スタートX回前』のペア列が1つも見つかりません。")

    status_cols = [f"ステータス{i}回前" for i in common_nums]
    start_cols  = [f"スタート{i}回前"   for i in common_nums]

    # 累計通常ゲーム数カラム存在チェック（書き込み先）
    target_col = "累計通常ゲーム数時短抜き"
    if target_col not in table_cols:
        raise RuntimeError(f"書き込み先カラム '{target_col}' が result_table に存在しません。")

    # 本日データの取得（まず日付を判定するため基本列だけ読む）
    df_basic = pd.read_sql_query(
        f'''
        SELECT ROWID AS rowid, {", ".join([f"[{c}]" for c in BASE_NEEDED])}
          FROM result_table
        ''',
        conn
    )

# 型整備
df_basic["実行日"] = pd.to_datetime(df_basic["実行日"], errors="coerce")
df_basic = df_basic.dropna(subset=["実行日"])

# 当日（テーブル内の最新“日付”）を特定
max_date = df_basic["実行日"].dt.date.max()
df_today_basic = df_basic[df_basic["実行日"].dt.date == max_date].copy()

# 当日の各台の「最新行のrowid」（更新対象はこの1件のみ）
latest_rowid_by_tai = (
    df_today_basic.sort_values(["台番号", "実行日", "rowid"], ascending=[True, False, False])
                  .groupby("台番号", as_index=False)
                  .agg(latest_rowid=("rowid", "max"))
)

# --- 本当に更新に必要な列だけで当日分を再取得（評価列＋書き込み先を含む） ---
select_cols = ["ROWID AS rowid"] + BASE_NEEDED + status_cols + start_cols + [target_col]
select_cols_sql = ", ".join([c if c.startswith("ROWID") else f"[{c}]" for c in select_cols])

with sqlite3.connect(db_path) as conn:
    df_today = pd.read_sql_query(
        f'''
        SELECT {select_cols_sql}
          FROM result_table
         WHERE date([実行日]) = ?
        ''',
        conn,
        params=(str(max_date),)
    )

# 当日の各台「最新行」のみ抽出
df_latest = df_today[df_today["rowid"].isin(latest_rowid_by_tai["latest_rowid"])].copy()

# クリーニング：スタート列は数値、ステータスは文字列で比較
for c in start_cols:
    df_latest[c] = pd.to_numeric(df_latest[c], errors="coerce").fillna(0)

for c in status_cols:
    # None/NaN 対応しつつ文字列化・トリム
    df_latest[c] = df_latest[c].astype(str).str.strip()

# 合計計算：「ステータスi回前 == '初当り'」のときだけ「スタートi回前」を加算
total = pd.Series(0, index=df_latest.index, dtype="float64")
for i in common_nums:
    s_col = f"ステータス{i}回前"
    st_col = f"スタート{i}回前"
    mask = (df_latest[s_col] == "初当り")
    total = total.add(df_latest[st_col].where(mask, 0), fill_value=0)

# intに丸め（必要なら）
df_latest[target_col] = total.fillna(0).astype(int)

# DB更新（当日・各台の最新行のみ）
updated = 0
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()
    for _, r in df_latest.iterrows():
        cur.execute(
            f'''
            UPDATE result_table
               SET [{target_col}] = ?
             WHERE ROWID = ?
               AND date([実行日]) = ?
            ''',
            (int(r[target_col]), int(r["rowid"]), str(max_date))
        )
        updated += cur.rowcount
    conn.commit()

print(f"✅ 累計通常ゲーム数時短抜き 更新完了: {updated} 行（当日各台の最新行のみ）")
print(f"[INFO] 所要時間: {time.time() - start_time:.2f} 秒")


[INFO] 使用DB: C:\Users\stray\myenv310\iwakuni-tekisasu-p\db\output.db
✅ 累計通常ゲーム数時短抜き 更新完了: 10 行（当日各台の最新行のみ）
[INFO] 所要時間: 0.09 秒
